[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/ai-agents-certified/notebooks/day-07-multi-agent-supervisor.ipynb#scrollTo=a1b2c3d4)

---
# Day 7 · Multi-Agent Supervisor Pattern — Router, Handoffs, and Subagents
**certified-journeys / ai-agents-certified** · Day 7 · Architecture

> **Goal for today:** Build a supervisor graph that routes between a research agent and a writer agent, using a FINISH sentinel to halt execution, with private state namespaces so subagents cannot overwrite each other's working memory.


In [ ]:
%pip install -q langgraph langchain-core langchain-openai python-dotenv


## Step 1 · Multi-Agent Architecture Overview

LangGraph supports two primary multi-agent topologies:

| Pattern | Shape | When to use |
|---------|-------|-------------|
| **Supervisor** | One central router node → subagent nodes | Coordinated tasks with specialization |
| **Swarm** | Agents hand off directly to each other | Decentralized peer routing |
| **Sequential** | A → B → C (no branching) | Pipeline / ETL workflows |

In the **supervisor pattern**:
- A lightweight router LLM reads the conversation state and outputs _only_ the next agent name (or `FINISH`).
- Each subagent does its specialized work and returns control to the supervisor.
- The cycle repeats until the supervisor outputs `FINISH`.

Key insight: The supervisor is a **conditional edge** disguised as a node. It never produces user-visible output — it only decides who speaks next.


In [ ]:
import os
from typing import Annotated, Literal, TypedDict
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# Use a stub LLM key for demonstration — replace with your key or set OPENAI_API_KEY
os.environ.setdefault("OPENAI_API_KEY", "sk-placeholder")

# ── Shared state schema ───────────────────────────────────────────────────────
# add_messages reducer appends to the list rather than overwriting it.
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]   # shared conversation thread
    next: str                                  # supervisor routing decision

print("State schema defined: AgentState with messages + next routing field")
print("Fields:", list(AgentState.__annotations__.keys()))


**What just happened?**
- We defined `AgentState` as a `TypedDict` — LangGraph uses the type annotations to know how to merge state updates from parallel or sequential nodes.
- **`add_messages`** is a built-in reducer: when two nodes both add to `messages`, the lists are concatenated, not overwritten. This is the default safe behaviour for chat history.
- The `next` field is a plain string — the supervisor writes it, and the conditional edge reads it to decide where to route.


## Step 2 · Supervisor Router Node

The supervisor LLM's job is minimal: read the conversation and output one word — the name of the next agent, or `FINISH`. A short, focused system prompt prevents hallucinated agent names.

**Router design rules:**
1. List available agents explicitly in the system prompt.
2. Ask for exactly one word in response.
3. Parse defensively — strip whitespace and lowercase before matching.


In [ ]:
# Define the available agents the supervisor can route to
MEMBERS = ["researcher", "writer"]
OPTIONS = MEMBERS + ["FINISH"]

SUPERVISOR_PROMPT = f"""You are a supervisor coordinating a team of agents: {MEMBERS}.
Given the conversation, decide which agent should act next, or output FINISH if the task is complete.
Respond with exactly one word — the agent name or FINISH. No punctuation, no explanation.
Available choices: {OPTIONS}"""

# ── Typed output so we can use .structured_output ────────────────────────────
from pydantic import BaseModel

class RouterOutput(BaseModel):
    next: Literal["researcher", "writer", "FINISH"]

def make_supervisor(llm):
    """Return a supervisor node function bound to the given LLM."""
    router_llm = llm.with_structured_output(RouterOutput)

    def supervisor_node(state: AgentState) -> dict:
        messages = [SystemMessage(content=SUPERVISOR_PROMPT)] + state["messages"]
        result: RouterOutput = router_llm.invoke(messages)
        return {"next": result.next}  # only update the routing field

    return supervisor_node

print("Supervisor factory defined")
print("RouterOutput schema:", RouterOutput.model_fields)


**What just happened?**
- **`with_structured_output(RouterOutput)`** forces the LLM to return a typed `RouterOutput` object rather than free text — this eliminates hallucinated agent names at the schema level.
- The supervisor node only writes to `next`; it never touches `messages`. This separation is important: the shared message thread is written only by subagents.
- **`Literal["researcher", "writer", "FINISH"]`** — Pydantic validates the value at parse time, so if the model tries to invent a new agent name, the call raises instead of silently routing to a nonexistent node.


## Step 3 · Private State Namespaces for Subagents

Without namespacing, two subagents writing to the same key can overwrite each other's work. The solution is to give each subagent its own typed sub-state field:

```
AgentState
├── messages        (shared, append-only)
├── next            (supervisor routing)
├── research_notes  (owned by researcher — others must not write here)
└── draft_text      (owned by writer — others must not write here)
```

This is a naming convention enforced by discipline, not by LangGraph itself. The real protection is: each subagent's node function only returns its own keys.


In [ ]:
from typing import Optional

# ── Extended state with private namespace fields ──────────────────────────────
class FullAgentState(TypedDict):
    messages:       Annotated[list, add_messages]
    next:           str
    research_notes: Optional[str]   # owned by researcher node
    draft_text:     Optional[str]   # owned by writer node


def researcher_node(state: FullAgentState) -> dict:
    """Research subagent — simulated (no real LLM call needed for structure demo)."""
    last_msg = state["messages"][-1].content if state["messages"] else "unknown topic"

    # In production, this calls an LLM or search tool. Here we simulate the output.
    notes = f"Research notes for: '{last_msg}'\n- Key fact 1\n- Key fact 2\n- Key fact 3"

    # ONLY write to research_notes and messages — never touch draft_text
    return {
        "research_notes": notes,
        "messages": [AIMessage(content=f"[Researcher] Gathered notes: {notes}", name="researcher")],
    }


def writer_node(state: FullAgentState) -> dict:
    """Writer subagent — reads research_notes, produces draft_text."""
    notes = state.get("research_notes", "No notes available")

    # ONLY write to draft_text and messages — never touch research_notes
    draft = f"Draft based on research:\n{notes}\n\n[Written summary paragraph here]"
    return {
        "draft_text": draft,
        "messages": [AIMessage(content=f"[Writer] Draft ready: {draft[:80]}…", name="writer")],
    }


print("Subagents defined with private state namespaces")
print("researcher_node writes: research_notes, messages")
print("writer_node writes:     draft_text, messages")


**What just happened?**
- Each subagent node returns **only** the keys it owns plus its contribution to `messages`. This is the private namespace pattern.
- The `name=` parameter on `AIMessage` lets you trace which subagent produced which message in the shared thread — essential for debugging multi-agent flows.
- **`state.get("research_notes", ...)`** — defensive access. On the first run, `research_notes` may be `None` if the writer is called before the researcher. The default prevents a `KeyError`.


## Step 4 · Graph Assembly and FINISH Sentinel

The `FINISH` sentinel is just a string the supervisor can output. We map it to `END` in the conditional routing function:

```
supervisor
   ├── "researcher" → researcher_node → supervisor
   ├── "writer"     → writer_node     → supervisor
   └── "FINISH"     → END
```

Every subagent routes back to the supervisor after completing — this creates the routing loop.


In [ ]:
from langgraph.graph import StateGraph, END

def build_supervisor_graph(llm):
    """Assemble and compile the full supervisor graph."""
    supervisor = make_supervisor(llm)

    builder = StateGraph(FullAgentState)

    # Add nodes
    builder.add_node("supervisor", supervisor)
    builder.add_node("researcher", researcher_node)
    builder.add_node("writer",     writer_node)

    # Conditional router: supervisor.next determines the edge
    def route(state: FullAgentState) -> str:
        return state["next"]  # "researcher" | "writer" | "FINISH"

    builder.add_conditional_edges(
        "supervisor",
        route,
        {
            "researcher": "researcher",
            "writer":     "writer",
            "FINISH":     END,          # FINISH maps to the built-in END node
        }
    )

    # All subagents loop back to supervisor
    builder.add_edge("researcher", "supervisor")
    builder.add_edge("writer",     "supervisor")

    # Entry point
    builder.set_entry_point("supervisor")

    return builder.compile()


# Build with a mock LLM for testing without a real API key
from unittest.mock import MagicMock

mock_llm = MagicMock()
mock_router = MagicMock()

# Simulate: researcher first, then writer, then FINISH
mock_router.invoke.side_effect = [
    RouterOutput(next="researcher"),
    RouterOutput(next="writer"),
    RouterOutput(next="FINISH"),
]
mock_llm.with_structured_output.return_value = mock_router

graph = build_supervisor_graph(mock_llm)
print("Graph compiled successfully")
print("Nodes:", list(graph.get_graph().nodes.keys()))


**What just happened?**
- **`add_conditional_edges`** reads the `state["next"]` key written by the supervisor node and routes to the matching node name.
- Mapping `"FINISH": END` is the canonical pattern — `END` is LangGraph's built-in terminal node, not a custom node you define.
- The mock LLM replays a fixed sequence (researcher → writer → FINISH), which lets us test graph topology without needing an API key.


## Step 5 · End-to-End Run with Multi-Step Handoffs

A real multi-step question requires at least two handoffs. Here we trace each step of the execution to confirm the routing loop is working correctly.


In [ ]:
# Run the graph and trace every state update
initial_state = {
    "messages": [HumanMessage(content="Research the history of quantum computing and write a summary.")],
    "next": "",
    "research_notes": None,
    "draft_text": None,
}

print("=" * 60)
print("Running supervisor graph...")
print("=" * 60)

step_count = 0
final_state = None

# stream() yields (node_name, state_update) tuples at each step
for step in graph.stream(initial_state, stream_mode="updates"):
    step_count += 1
    node_name, updates = next(iter(step.items()))
    print(f"\nStep {step_count}: [{node_name}]")

    if "next" in updates:
        print(f"  Supervisor decision → {updates['next']}")
    if "research_notes" in updates and updates["research_notes"]:
        print(f"  research_notes set: {updates['research_notes'][:60]}…")
    if "draft_text" in updates and updates["draft_text"]:
        print(f"  draft_text set: {updates['draft_text'][:60]}…")
    if "messages" in updates:
        for msg in updates["messages"]:
            print(f"  msg [{getattr(msg, 'name', type(msg).__name__)}]: {msg.content[:80]}")

    final_state = step

print("\n" + "=" * 60)
print(f"Completed in {step_count} steps")
print("Handoff sequence: Human → supervisor → researcher → supervisor → writer → supervisor(FINISH) → END")


**What just happened?**
- **`stream_mode="updates"`** yields one dict per node execution containing only the keys that node changed — this is the cleanest way to trace which node is responsible for each state mutation.
- We confirmed **3 supervisor invocations** (→ researcher, → writer, → FINISH) and 2 subagent executions — exactly 2 handoffs as required.
- The `research_notes` and `draft_text` keys were written by separate nodes and never collided, confirming the private namespace pattern worked.


## Step 6 · Edge Case: Empty Results from Subagents

What happens when a subagent returns no useful output? Without a guard, the writer will draft from `None` and the supervisor may loop forever.

**Fix:** Add a `retry_count` field to state. If a subagent's output is empty, the supervisor checks the count and routes to FINISH after N retries instead of looping.


In [ ]:
class RobustAgentState(TypedDict):
    messages:       Annotated[list, add_messages]
    next:           str
    research_notes: Optional[str]
    draft_text:     Optional[str]
    retry_count:    int          # tracks consecutive empty-result turns

MAX_RETRIES = 2  # if both agents return empty twice, FINISH anyway

ROBUST_SUPERVISOR_PROMPT = f"""{SUPERVISOR_PROMPT}
If the previous agent returned empty or unhelpful results and retry_count >= {MAX_RETRIES},
output FINISH to prevent infinite loops."""

def researcher_node_robust(state: RobustAgentState) -> dict:
    """Researcher that can return empty to simulate failure."""
    last_msg = state["messages"][-1].content if state["messages"] else ""
    # Simulate: if the question is 'EMPTY TEST', return nothing
    if "EMPTY TEST" in last_msg:
        return {
            "research_notes": "",  # empty result
            "retry_count": state.get("retry_count", 0) + 1,
            "messages": [AIMessage(content="[Researcher] No results found.", name="researcher")],
        }
    notes = f"Research complete for: {last_msg[:50]}"
    return {
        "research_notes": notes,
        "retry_count": 0,  # reset on success
        "messages": [AIMessage(content=f"[Researcher] Done: {notes}", name="researcher")],
    }

print("Robust researcher with retry_count guard defined")
print(f"System will FINISH after {MAX_RETRIES} consecutive empty results")

# Verify the guard logic with a mock state
test_state: RobustAgentState = {
    "messages": [HumanMessage(content="EMPTY TEST")],
    "next": "",
    "research_notes": None,
    "draft_text": None,
    "retry_count": 0,
}
result = researcher_node_robust(test_state)
print(f"\nAfter empty result: retry_count = {result['retry_count']} (was 0)")
print(f"research_notes = '{result['research_notes']}'")
print("→ Supervisor would see retry_count=1 and notes='' → route to writer or retry")
print(f"→ After {MAX_RETRIES} retries, supervisor should output FINISH")


**What just happened?**
- **`retry_count`** is a plain integer in state — no special reducer needed because only one node writes it at a time.
- Resetting to `0` on success prevents old retry counts from carrying over between unrelated subtasks in the same conversation.
- The supervisor's system prompt is the right place to express the retry policy — it keeps the routing logic in one place and avoids hardcoding `if retry_count >= N` in the graph topology.


In [ ]:
# Challenge: Multi-step research question requiring ≥ 2 handoffs
#
# Your task: Extend the supervisor graph to include a third subagent: 'editor'.
# The editor reviews draft_text and produces a final polished version.
#
# Scaffold:
# 1. Add an 'editor_output' field to FullAgentState.
# 2. Implement editor_node(state) that reads draft_text and returns editor_output.
# 3. Update RouterOutput Literal to include 'editor'.
# 4. Re-build the graph with the new node and edge (editor → supervisor).
# 5. Test with a mock LLM that routes: supervisor→researcher→supervisor→writer→supervisor→editor→supervisor→FINISH.
#    Confirm 3 handoffs in the trace output.
#
# Expected output:
#   Step 1: [supervisor] → researcher
#   Step 2: [researcher] sets research_notes
#   Step 3: [supervisor] → writer
#   Step 4: [writer] sets draft_text
#   Step 5: [supervisor] → editor
#   Step 6: [editor] sets editor_output
#   Step 7: [supervisor] → FINISH

# Your solution here


---
## Day 7 key concepts recap

| Concept | What to remember |
|---|---|
| Supervisor pattern | One router LLM writes `next`; conditional edge routes to subagent or END |
| FINISH sentinel | Map `"FINISH": END` in `add_conditional_edges` — do not create a FINISH node |
| Private namespaces | Each subagent only writes its own keys; shared state is append-only messages |
| `with_structured_output` | Forces typed output — eliminates hallucinated agent names |
| `stream_mode="updates"` | Yields per-node diffs — cleaner than full state snapshots for tracing |
| Retry guard | `retry_count` in state + check in supervisor prompt prevents infinite loops |

> **Tip:** The supervisor's router LLM only needs to output a single token: the next agent's name or FINISH. Keep its system prompt to one paragraph — complexity in the router causes hallucinated handoffs.

---
## What's next
**Day 8** → Streaming and Long-Term Memory — stream tokens to the terminal as they arrive, and persist user preferences across sessions with `InMemoryStore`.

Mark Day 7 complete in your [tracker](../index.html).
